In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/processed_jobs.csv")
df.head()

,title,location,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,has_salary,text_length
0,Marketing Intern,"US, NY, New York",NaN,"<h3>We're Food52, and we've created a groundbr...","<p>Food52, a fast-growing, James Beard Award-w...",<ul>\r\n<li>Experience with content management...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0,0,1002
1,Customer Service - Cloud Video Production,"NZ, , Auckland",NaN,"<h3>90 Seconds, the worlds Cloud Video Product...",<p>Organised - Focused - Vibrant - Awesome!<br...,<p><b>What we expect from you:</b></p>\r\n<p>Y...,<h3><b>What you will get from us</b></h3>\r\n<...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0,0,2175
2,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,<h3></h3>\r\n<p>Valor Services provides Workfo...,"<p>Our client, located in Houston, is actively...",<ul>\r\n<li>Implement pre-commissioning and co...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0,0,362
3,Account Executive - Washington DC,"US, DC, Washington",NaN,<p>Our passion for improving quality of life t...,<p><b>THE COMPANY: ESRI – Environmental System...,<ul>\r\n<li>\r\n<b>EDUCATION: </b>Bachelor’s o...,<p>Our culture is anything but corporate—we ha...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0,0,2858
4,Bill Review Manager,"US, FL, Fort Worth",NaN,<p>SpotSource Solutions LLC is a Global Human ...,<p><b>JOB TITLE:</b> Itemization Review Manage...,<p><b>QUALIFICATIONS:</b></p>\r\n<ul>\r\n<li>R...,<p>Full Benefits Offered</p>,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0,0,1872


In [2]:
text_cols = ['title', 'company_profile', 'description', 'requirements', 'benefits']

for col in text_cols:
    df[col] = df[col].fillna('')

df['text'] = (
    df['title'] + ' ' +
    df['company_profile'] + ' ' +
    df['description'] + ' ' +
    df['requirements'] + ' ' +
    df['benefits']
)

In [3]:
from sklearn.model_selection import train_test_split

X = df['text']
y = df['fraudulent']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Используется stratified split, чтобы сохранить исходное соотношение классов в train и test.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

baseline_model.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, stop_words='english')),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

In [5]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

y_pred = baseline_model.predict(X_test)
y_proba = baseline_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.97      0.98      3403
           1       0.59      0.89      0.71       173

    accuracy                           0.97      3576
   macro avg       0.79      0.93      0.85      3576
weighted avg       0.97      0.97      0.97      3576

ROC-AUC: 0.9830071732014765
Confusion matrix:
[[3298  105]
 [  19  154]]


Результаты baseline-модели:
В качестве baseline была использована модель TF-IDF + Logistic Regression.
Модель показала высокое качество:
ROC-AUC: 0.98
F1-score для класса мошеннических вакансий: 0.71

Анализ качества:
Модель демонстрирует высокий recall (0.89) для мошеннических вакансий, что свидетельствует о способности обнаруживать большинство мошеннических объявлений.
При этом значение precision (0.59) ниже, что указывает на наличие ложноположительных срабатываний (модель иногда ошибочно классифицирует реальные вакансии как мошеннические).

Вывод:
В условиях сильного дисбаланса классов приоритетной является способность модели обнаруживать мошеннические вакансии.
Поэтому высокий recall является более важным показателем, чем precision, так как пропуск мошеннического объявления более критичен, чем ложное срабатывание.

In [6]:
import joblib

joblib.dump(baseline_model, "../models/baseline_tfidf_logreg.joblib")

['../models/baseline_tfidf_logreg.joblib']

In [7]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

rf_model.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, stop_words='english')),
                ('clf', RandomForestClassifier(random_state=42))])

In [8]:
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3403
           1       1.00      0.61      0.76       173

    accuracy                           0.98      3576
   macro avg       0.99      0.80      0.87      3576
weighted avg       0.98      0.98      0.98      3576

ROC-AUC: 0.9823829365113067


Несмотря на более высокий F1-score, модель RandomForest демонстрирует низкий recall, что означает, что она пропускает значительную часть мошеннических вакансий.
В то же время baseline-модель (Logistic Regression) показывает более высокий recall (0.89), что делает её более подходящей для данной задачи.

Вывод:
В рамках задачи детекции мошеннических вакансий более предпочтительной является модель Logistic Regression, так как она лучше справляется с обнаружением мошеннических объявлений, несмотря на большее количество ложных срабатываний.

Замечание по обработке текста:

В рамках данного этапа (CP1) использовалась базовая обработка текстовых данных без применения сложных методов NLP (лемматизация, удаление стоп-слов вручную и т.д.).

Это сделано осознанно, так как целью данного этапа является построение baseline-модели и проведение первичных экспериментов.